# ☀️ SunSight: Solar Radiation Forecasting

**Track:** AI + Climate

**Team:** Karthick Prakhul and Siddharth Vadivel

**One-sentence pitch:** SunSight helps grid operators and renewable energy planners by taking historical weather data and predicting next-day solar radiation to optimize power distribution and battery storage.

**Area:** Singapore

## Loading the data and libraries


In [ ]:
# Core data, model, and visualization libraries
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt

### The dataset contains historical daily weather observations and solar radiation values for Singapore. We load it from `open_meteo.csv` into a pandas DataFrame, then inspect the first and last rows to check that it was read correctly and understand its date coverage.

In [ ]:
DATA_PATH = '/content/open_meteo.csv'

om_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(om_df)} rows from {DATA_PATH}")
display(om_df.head(3))
display(om_df.tail(3))

## Engineering the Target Variable (`shift(-1)`)
* **Why we do this:** To predict the *future* (tomorrow's solar radiation), our model needs to look at today's weather inputs and connect them to tomorrow's output.
* **The Technique:** We use Pandas `.shift(-1)` to slide tomorrow's radiation value up into today's row. This aligns today's weather features with tomorrow's actual solar outcome.

In [ ]:
# Prepare date values and align today's weather with tomorrow's radiation.
RAD_COL = 'Solar Radiation (MJ/m²)'
REQUIRED_COLUMNS = {'Day', RAD_COL, 'Temperature (°C)', 'Sunshine Duration (s)', 'Precipitation (mm)'}

missing_columns = REQUIRED_COLUMNS.difference(om_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

om_df = om_df.copy()
om_df['Day'] = pd.to_datetime(om_df['Day'], format='%d/%m/%y', errors='raise')
om_df = om_df.sort_values('Day').drop_duplicates(subset='Day', keep='last')
om_df = om_df.reset_index(drop=True)

# Tomorrow's radiation becomes today's target; the final row has no next-day observation.
om_df['Target'] = om_df[RAD_COL].shift(-1)
om_df = om_df.dropna(subset=['Target']).reset_index(drop=True)

print(f"Prepared {len(om_df)} forecast records")
display(om_df.head(3))
display(om_df.tail(3))

## Chronological Train/Test Split
* **Why chronological (not random)?** In time-series forecasting, random splitting allows the model to "cheat" by training on future data to predict the past.
* **Our Strategy:** We split the data chronologically—training on the first 80% of days and testing on the final 20% (the most recent data) to simulate a realistic, real-world deployment scenario.

In [ ]:
# Split chronologically so the test set contains only unseen, later observations.
TRAIN_RATIO = 0.80
train_size = int(len(om_df) * TRAIN_RATIO)
train_df = om_df.iloc[:train_size].copy()
test_df = om_df.iloc[train_size:].copy()

# Weather known today is used to forecast tomorrow's radiation.
FEATURE_COLUMNS = [
    'Temperature (°C)',
    'Sunshine Duration (s)',
    'Precipitation (mm)',
]

X_train = train_df[FEATURE_COLUMNS]
X_test = test_df[FEATURE_COLUMNS]
y_train = train_df['Target']
y_test = test_df['Target']

print(f"Training rows: {len(train_df)} | Test rows: {len(test_df)}")

## Baseline Comparison & Model Training
* **The Baseline ("Persistence Model"):** We assume tomorrow's weather will be identical to today's. This gives us a benchmark error score that our AI model *must* beat to prove it learned something useful.
* **Linear Regression:** We train a linear model on key weather features (radiation, sunshine duration, temperature, and precipitation) to see if mathematical patterns can outperform the naive baseline.

In [ ]:
# Persistence baseline: assume tomorrow's radiation equals today's radiation.
baseline_preds = test_df[RAD_COL]

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = mean_squared_error(y_test, baseline_preds) ** 0.5
baseline_r2 = r2_score(y_test, baseline_preds)

# Standardize inputs before fitting, improving numerical stability and coefficient comparison.
model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression()),
])
model.fit(X_train, y_train)

model_preds = model.predict(X_test)
model_mae = mean_absolute_error(y_test, model_preds)
model_rmse = mean_squared_error(y_test, model_preds) ** 0.5
model_r2 = r2_score(y_test, model_preds)

print("Baseline (Persistence)")
print(f"  MAE: {baseline_mae:.4f} | RMSE: {baseline_rmse:.4f} | R²: {baseline_r2:.4f}")
print("\nLinear Regression")
print(f"  MAE: {model_mae:.4f} | RMSE: {model_rmse:.4f} | R²: {model_r2:.4f}")

if model_mae < baseline_mae:
    print("\nSuccess: the model has a lower mean absolute error than the baseline.")
else:
    print("\nReview needed: the model did not beat the persistence baseline on MAE.")

## Visualizing Model Predictions vs. Actuals
* **Why visualization matters:** While metrics like MAE give us a numerical score, seeing visual proof is more convincing. A time-series graph lets everyone see instantly how closely our model's predictions track actual solar radiation peaks and valleys.
* **Our Strategy:** We use `matplotlib` to plot our model's predictions against the actual test data over time.

In [ ]:
# Plot actual and predicted values on the test timeline.
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    test_df['Day'], y_test,
    label='Actual Radiation', color='tab:orange', alpha=0.85, linewidth=2,
)
ax.plot(
    test_df['Day'], model_preds,
    label='Model Predictions', color='royalblue', linestyle='--', alpha=0.95, linewidth=2,
)

ax.set_title('Solar Radiation Forecast: Actual vs. Linear Regression', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Solar Radiation (MJ/m²)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.45)
fig.tight_layout()
plt.show()

## Model Explainability (Feature Coefficients)
* **Why this matters:** Linear regression is a white-box model. Unlike a black-box neural network, we can look directly inside the math to see which weather features had the biggest impact on predicting tomorrow's solar radiation.
* **Our Strategy:** We extract and plot the model's coefficients to show whether variables like sunshine duration or temperature drive solar output up or down.

In [ ]:
# Extract standardized coefficients to compare each weather feature's relative impact.
coefficients = model.named_steps['regressor'].coef_
coeff_df = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Coefficient': coefficients,
}).assign(AbsoluteCoefficient=lambda frame: frame['Coefficient'].abs())
coeff_df = coeff_df.sort_values('AbsoluteCoefficient', ascending=False)

print("Feature Coefficients (Impact on Tomorrow's Radiation):")
print(coeff_df[['Feature', 'Coefficient']].to_string(index=False))

## **Conclusion**
* **What we accomplished:** Built an end-to-end, chronological solar radiation forecasting pipeline with data validation, a persistence baseline, and an explainable linear model.
* **Key metric:** MAE is the primary comparison metric because it reports average absolute forecasting error in the same units as radiation.
* **Key takeaway:** The model is useful only when it improves on the persistence baseline on unseen test data.
